In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE



In [4]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

X = train.drop("target", axis=1)
y = train["target"]

X_test_original = test



In [5]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
#splitting the training data set into train and validation set,...random_state=42 means same split 
#every run, random state=None--different random split every run...stratify y to make sure 
#train and validation sets keep the same class proportion..if dataset has 90% class 0 and 10% class 1 
#the split should be 90/10 

In [6]:
scaler = StandardScaler() #x-mean/std..basically normalization
X_train = scaler.fit_transform(X_train) #we only normalize the training data ..not the validation set or the test set
X_val   = scaler.transform(X_val) #we only apply the same scaling as applied to the train data, but we dont fit the scaling in the data..to avoid data leakage
X_test  = scaler.transform(X_test_original)#same as validation set 


In [7]:
sm = SMOTE(random_state=42) #synthetic minority oversampling technique..random state =42 ensures reproducibility
X_train_res, y_train_res = sm.fit_resample(X_train, y_train) #fit() makes sure how minority samples look..resample() creates new synthetic samples 
print("After SMOTE:", np.bincount(y_train_res)) #prints the no.of samples in each class after applying SMOTE 
# only apply smote in train data..not in validation or test dataset 


After SMOTE: [1440 1440]


In [8]:
pos = y.sum()  #as the classes are 0 and 1..so taking sumation gives how many 1's are there in the dataset (positive samples)
neg = len(y) - pos # tells the no.of 0's in the dataset (negative samples)
scale_pos_weight = neg / pos #ratio important when applying XGboost ..it implies each positive samples get 9 times more weight..it hellps in balancing the classes..and detection of minority classes much easier 
print("Scale Pos Weight =", scale_pos_weight)



Scale Pos Weight = 9.0


In [9]:
xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight, #gives more weight to the minority class
    n_estimators=600, #no.of trees
    max_depth=7, #tree depth 
    learning_rate=0.05, 
    subsample=0.8, #% of rows used per tree
    colsample_bytree=0.8, #% of columns used per tree
    random_state=42, #makes the model more reproducible 
    n_jobs=-1 # uses all the cpu core in the computer..in hp victus 2048 cores are present..-1 ensures all the cpu cores are used
)


In [10]:
xgb.fit(X_train_res, y_train_res) #x and y data after balanced SMOTE



,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [11]:
y_prob_val = xgb.predict_proba(X_val)[:, 1]  #predicvts probabibility of the positive class(1)..selects the 2nd column ..no need for prob of class 0..as sumation of probs  equates to 1
print("First 20 probabilities:", y_prob_val[:20]) # we need to fine tune the parameters properly..so we use y_prob_val ..and not the test dataset 

First 20 probabilities: [3.1203756e-04 1.1307016e-04 6.8515092e-01 1.3783914e-03 1.2668999e-03
 5.1504473e-04 3.3784371e-03 2.3596251e-04 3.1326090e-05 8.1790218e-05
 7.8783296e-02 4.5082840e-04 9.9552864e-01 2.6743284e-03 7.7970373e-04
 7.7893688e-05 9.7336316e-01 9.9758697e-01 4.1550407e-04 2.0834306e-04]


In [ ]:
#thresholds = np.arange(0.05, 2, 0.01)
#best_t = 0.5
#best_bal = 0

#for t in thresholds:
 #   preds = (y_prob_val >= t).astype(int)  #if predict>= thresholds..predict 1,otherwise 0
  ## if bal > best_bal:
    #    best_bal = bal
     #   best_t = t

#print("BEST THRESHOLD =", best_t)
#print("BEST BALANCED ACCURACY =", best_bal)



NameError: name 'np' is not defined

In [12]:
X_scaled_full = scaler.fit_transform(X)
X_full_res, y_full_res = sm.fit_resample(X_scaled_full, y)

xgb.fit(X_full_res, y_full_res)  #after fine tuning the parameteres..we now apply the normalization to the entire training data set given to us(original+synthetic) 


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [13]:
X_test_scaled = scaler.transform(X_test_original)
y_test_prob = xgb.predict_proba(X_test_scaled)[:, 1]

# apply best threshold
y_test_pred = (y_test_prob >=0.9).astype(int)  #converts probabilities into hard class predictions 0 or 1


In [14]:
submission = pd.DataFrame({
    "id": test["id"],
    "target": y_test_pred
})

submission.to_csv("submission.csv", index=False)
print("submission.csv created successfully!")

submission.csv created successfully!
